# Socceraction

## Libraries and Constants

In [1]:
# Import libraries
import pandas as pd
import socceraction.vaep.features as fs
import xgboost as xgb

from pathlib import Path
from sklearn.metrics import brier_score_loss, roc_auc_score, log_loss
from tqdm import tqdm

In [2]:
# Ignore warnings
import warnings

warnings.filterwarnings(action="ignore", category=FutureWarning)
warnings.filterwarnings(action="ignore", message="Inferred xy_fidelity_version=2.", category=UserWarning)
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

In [3]:
# Define the data directory
DATA_DIR = Path("../data")
SPADL_DIR = Path("data")

SPADL_H5 = SPADL_DIR / "spadl-statsbomb.h5"
FEATURES_H5 = SPADL_DIR / "features.h5"
LABELS_H5 = SPADL_DIR / "labels.h5"
PREDICTIONS_H5 = SPADL_DIR / "predictions.h5"

## VAEP

In [4]:
# Load the games data from the HDF5 file
games_df = pd.read_hdf(SPADL_H5, "games")
print(f"Number of games: {len(games_df)}")

Number of games: 380


In [5]:
# Define the training and testing data
# Only for the purpose of this example we will use the same data for training and evaluation
train_games = games_df
test_games = games_df

In [ ]:
# Create the X and Y sets, which are the features and labels respectively
xfns = [
    fs.actiontype,
    fs.actiontype_onehot,
    # fs.actiontype_result_onehot
    fs.result,
    fs.result_onehot,
    # fs.bodypart,
    fs.bodypart_onehot,
    fs.startlocation,
    fs.endlocation,
    fs.startpolar,
    fs.endpolar,
    fs.movement,
    fs.space_delta,
    # fs.time,
    fs.time_delta,
    fs.team,
    fs.goalscore,
]


def getXY(games):
    # 1. Select feature set X
    Xcols = fs.feature_column_names(xfns, nb_prev_actions=1)
    X = []

    for game_id in tqdm(games.game_id, desc="Selecting features"):
        Xi = pd.read_hdf(FEATURES_H5, f"game_{game_id}")
        X.append(Xi[Xcols])

    X = pd.concat(X).reset_index(drop=True)

    # 2. Select label Y
    Ycols = ["scores", "concedes"]
    Y = []

    for game_id in tqdm(games.game_id, desc="Selecting labels"):
        Yi = pd.read_hdf(LABELS_H5, f"game_{game_id}")
        Y.append(Yi[Ycols])

    Y = pd.concat(Y).reset_index(drop=True)

    # Return the features and labels as X and Y
    return X, Y


# Get the features and labels for the training games
X, Y = getXY(train_games)
print("X:")
print(*list(X.columns), sep="\n")
print("Y:")
print(*list(Y.columns), sep="\n")

Selecting labels: 100%|██████████| 380/380 [00:01<00:00, 222.55it/s]


X:
actiontype_a0
actiontype_pass_a0
actiontype_cross_a0
actiontype_throw_in_a0
actiontype_freekick_crossed_a0
actiontype_freekick_short_a0
actiontype_corner_crossed_a0
actiontype_corner_short_a0
actiontype_take_on_a0
actiontype_foul_a0
actiontype_tackle_a0
actiontype_interception_a0
actiontype_shot_a0
actiontype_shot_penalty_a0
actiontype_shot_freekick_a0
actiontype_keeper_save_a0
actiontype_keeper_claim_a0
actiontype_keeper_punch_a0
actiontype_keeper_pick_up_a0
actiontype_clearance_a0
actiontype_bad_touch_a0
actiontype_non_action_a0
actiontype_dribble_a0
actiontype_goalkick_a0
result_a0
result_fail_a0
result_success_a0
result_offside_a0
result_owngoal_a0
result_yellow_card_a0
result_red_card_a0
bodypart_foot_a0
bodypart_head_a0
bodypart_other_a0
bodypart_head/other_a0
start_x_a0
start_y_a0
end_x_a0
end_y_a0
start_dist_to_goal_a0
start_angle_to_goal_a0
end_dist_to_goal_a0
end_angle_to_goal_a0
dx_a0
dy_a0
movement_a0
goalscore_team
goalscore_opponent
goalscore_diff
Y:
scores
concedes


## Model

In [7]:
# 3. Train classifiers F(X) = Y

Y_hat = pd.DataFrame()
models = {}
for col in list(Y.columns):
    model = xgb.XGBClassifier(n_estimators=50, max_depth=3, n_jobs=-3, verbosity=1, enable_categorical=True)
    model.fit(X, Y[col])
    models[col] = model

In [8]:
testX, testY = X, Y


def evaluate(y, y_hat):
    p = sum(y) / len(y)
    base = [p] * len(y)
    brier = brier_score_loss(y, y_hat)
    print(f"  Brier score: {brier:.5f} ({brier / brier_score_loss(y, base):.5f})")
    ll = log_loss(y, y_hat)
    print(f"  Log Loss score: {ll:.5f} ({ll / log_loss(y, base):.5f})")
    print(f"  ROC AUC: {roc_auc_score(y, y_hat):.5f}")


for col in testY.columns:
    Y_hat[col] = [p[1] for p in models[col].predict_proba(testX)]
    print(f"### Y: {col} ###")
    evaluate(testY[col], Y_hat[col])

### Y: scores ###
  Brier score: 0.00920 (0.85242)
  Log Loss score: 0.04732 (0.78694)
  ROC AUC: 0.82122
### Y: concedes ###
  Brier score: 0.00208 (0.98613)
  Log Loss score: 0.01428 (0.94402)
  ROC AUC: 0.79381


In [9]:
# Get rows with game_id per action
A = []
for game_id in tqdm(games_df.game_id, "Loading game ids"):
    Ai = pd.read_hdf(SPADL_H5, f"actions/game_{game_id}")
    A.append(Ai[["game_id"]])
A = pd.concat(A).reset_index(drop=True)

# Concatenate action game_id rows with predictions and save per game
grouped_predictions = pd.concat([A, Y_hat], axis=1).groupby("game_id")

with pd.HDFStore(PREDICTIONS_H5) as prediction_store:
    for k, df in tqdm(grouped_predictions, desc="Saving predictions per game"):
        df = df.reset_index(drop=True)
        prediction_store.put(f"game_{int(k)}", df[Y_hat.columns])

Saving predictions per game: 100%|██████████| 380/380 [00:01<00:00, 227.40it/s]
